In [1]:
import torch
import numpy as np
import pandas as pd 
import torch.nn as nn
from torchvision import transforms as transforms
from torch.utils.data import Dataset,DataLoader

 

In [2]:
train_transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])
test_transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

In [3]:
train = "/kaggle/input/datasets/puneet6060/intel-image-classification/seg_train/seg_train"
test = "/kaggle/input/datasets/puneet6060/intel-image-classification/seg_test/seg_test"




In [4]:
import os
from PIL import Image

class MyDataset(Dataset):
    def __init__(self, data, transform=None):
        super().__init__()
        self.data = data
        self.transform = transform
        self.classes = sorted(os.listdir(data))
        self.class_to_idx = {cls: i for i, cls in enumerate(self.classes)}
        
        self.samples = []
        for cls in self.classes:
            cls_folder = os.path.join(data, cls)
            for fname in os.listdir(cls_folder):
                self.samples.append((os.path.join(cls_folder, fname), self.class_to_idx[cls]))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        image = Image.open(img_path).convert("RGB")
        if self.transform:
            image = self.transform(image)
        return image, label



In [5]:
train_dataset = MyDataset(train, transform=train_transform)
test_dataset =MyDataset(test,transform=test_transform)

len(train_dataset)

14034

In [6]:
print(os.listdir(os.path.join(train, "buildings")))

['2193.jpg', '11378.jpg', '10730.jpg', '17338.jpg', '16730.jpg', '10924.jpg', '15653.jpg', '19674.jpg', '1786.jpg', '7222.jpg', '19901.jpg', '8623.jpg', '12551.jpg', '2936.jpg', '8914.jpg', '18672.jpg', '1501.jpg', '11855.jpg', '5477.jpg', '8342.jpg', '7544.jpg', '4398.jpg', '19407.jpg', '18335.jpg', '15030.jpg', '13377.jpg', '8444.jpg', '10598.jpg', '17973.jpg', '13400.jpg', '11428.jpg', '18854.jpg', '12068.jpg', '16575.jpg', '6894.jpg', '4979.jpg', '12770.jpg', '4351.jpg', '15794.jpg', '12999.jpg', '3670.jpg', '1161.jpg', '13624.jpg', '19776.jpg', '3645.jpg', '1940.jpg', '5852.jpg', '1009.jpg', '12540.jpg', '17983.jpg', '8592.jpg', '6190.jpg', '10849.jpg', '11024.jpg', '1760.jpg', '15261.jpg', '6605.jpg', '3364.jpg', '5998.jpg', '15404.jpg', '3310.jpg', '13287.jpg', '3209.jpg', '8367.jpg', '4812.jpg', '9037.jpg', '2319.jpg', '9135.jpg', '1539.jpg', '10500.jpg', '5817.jpg', '6213.jpg', '17211.jpg', '16733.jpg', '2041.jpg', '8380.jpg', '9364.jpg', '10383.jpg', '7292.jpg', '3743.jpg', '

In [7]:
print(os.listdir(train))

['mountain', 'street', 'buildings', 'sea', 'forest', 'glacier']


In [8]:
train_loader=DataLoader(train_dataset,batch_size=64,shuffle=True)
test_loader=DataLoader(test_dataset,batch_size=64,shuffle=False)

In [9]:
print(len(train_loader), len(test_loader))

220 47


In [10]:
print(len(train_dataset))
print(len(test_dataset))

14034
3000


In [11]:
print(len(train_loader), len(test_loader))

220 47


In [12]:
class CNN(nn.Module):
    def __init__(self,num_classes=6):
        super().__init__()
        self.Conv1=nn.Conv2d(in_channels=3,out_channels=32,kernel_size=3,padding=1)
        self.bn1=nn.BatchNorm2d(32)
        self.relu1=nn.ReLU()
        self.maxpol1=nn.MaxPool2d(2)
        #Block2:
        self.Conv2=nn.Conv2d(in_channels=32,out_channels=64,kernel_size=3,padding=1)
        self.bn2=nn.BatchNorm2d(64)
        self.relu2=nn.ReLU()
        self.maxpol2=nn.MaxPool2d(2)
        #Block3:
        self.Conv3=nn.Conv2d(in_channels=64,out_channels=128,kernel_size=3,padding=1)
        self.bn3=nn.BatchNorm2d(128)
        self.relu3=nn.ReLU()
        self.maxpol3=nn.MaxPool2d(2)
        #Block4:
        self.Conv4=nn.Conv2d(in_channels=128,out_channels=256,kernel_size=3,padding=1)
        self.bn4=nn.BatchNorm2d(256)
        self.relu4=nn.ReLU()
        self.maxpol4=nn.MaxPool2d(2)
        #Flatten:
        self.flatten=nn.Flatten()  
        self.linear1=nn.Linear(4096,256)
        self.relu5=nn.ReLU()
        self.dropout=nn.Dropout()
        self.linear2=nn.Linear(256,6)
    def __forward__(self,x):
        
         # Block 1
          x = self.Conv1(x)
          x = self.bn1(x)
          x = self.relu1(x)
          x = self.maxpol1(x)

         # Block 2
          x = self.Conv2(x)
          x = self.bn2(x)
          x = self.relu2(x)
          x = self.maxpol2(x)

        # Block 3
          x = self.Conv3(x)
          x = self.bn3(x)
          x = self.relu3(x)
          x = self.maxpol3(x)

        # Block 4
          x = self.Conv4(x)
          x = self.bn4(x)
          x = self.relu4(x)
          x = self.maxpol4(x)

        # Flatten
          x = self.flatten(x)

         # Fully connected layers
          x = self.linear1(x)
          x = self.relu5(x)
          x = self.dropout(x)
          x = self.linear2(x)
          return x


  